In [2]:
import sys
sys.path.append('../')
import time

import numpy as np
import matplotlib.pyplot as plt

import epics
from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

Initialize cax control of siriuspy devices

In [42]:
CAX = CAXCtrl()

# Scan

Parameters

In [ ]:
STEP = 0.0001 # [mm]
SCALE = 1     # [um/px]
DELAY = 5     # [s]

Functions

In [ ]:
def dvfs_acquire():
    CAX.dvf_A1.cmd_acquire_on()
    CAX.dvf_B1.cmd_acquire_on()

def current_parameters():
    return {
        'slit1': {
            'top': CAX.slit_A1.top_pos,
            'bottom': CAX.slit_A1.bottom_pos,
            'left': CAX.slit_A1.left_pos,
            'right': CAX.slit_A1.right_pos
        },
        'dvf1': {
            'expo_time': CAX.dvf_A1.acquisition_time
        },
        'dvf2': {
            'expo_time': CAX.dvf_A1.acquisition_time,
            'z_pos': CAX.dvf_B1.z_pos
        },
        'mirror': {
            'ry': CAX.mirror.ry_pos,
            'tx': CAX.mirror.tx_pos,
            'y1': CAX.mirror.y1_pos,
            'y2': CAX.mirror.y2_pos,
            'y3': CAX.mirror.y3_pos
        }
    }

def fwhm(data):
    threshold = 0.5*np.max(data)
    mask = data > threshold
    return np.sum(mask) * SCALE

def peak(data):
    return np.max(data)

def position(data):
    return np.argmax(data) * SCALE

def move(ry_pos,delay=DELAY):
    CAX.mirror.ry_pos = ry_pos
    time.sleep(delay)

def move_step(step=STEP,delay=DELAY):
    pos = CAX.mirror.ry_pos + step
    move(ry_pos=pos,delay=delay)

def table_line():

    img = CAX.dvf_B1.image
    
    projx = np.sum(img,axis=0)
    projy = np.sum(img,axis=1)

    return [np.round(CAX.mirror.ry_pos,8),
            position(projx), fwhm(projx), peak(projx),
            position(projy), fwhm(projy), peak(projy)]

Initial state

In [ ]:
parameters0 = current_parameters()

file_initial = 'initial_state_ry_scan_20250724.txt'

with open(file_initial,'w') as f:
    for key, value in parameters0.items():
        string = f'{key}: {value}\n\n'
        f.write(string)
        print(string)

Tests

In [ ]:
header = '#    Ry          X        fwhmx    peakx      Y        fwhmy   peaky'
table  = [table_line()]

In [ ]:

dRy = 0.000
move_step(step=dRy)

table.append(table_line())

file_table = 'table_ry_2025-07-24.dat'

f = open(file_table,mode='w')
f.write(header + '\n')

print(header)
for line in table:
    f.write(' '.join([str(ela) for ela in line]) + '\n')
    print(*line,sep='      ')

f.close()


#    Ry          X        fwhmx    peakx      Y        fwhmy   peaky
0.33626625      751      139      94451      844      84      147769
0.3363615      742      153      87717      844      85      148532
0.33645675      954      139      74628      838      85      116547
0.336552      779      162      76666      842      83      138998
0.33664725      769      143      88766      844      84      142704
0.3367425      798      138      85060      844      84      133406
0.33683775      818      139      82100      842      84      128824
0.336933      828      140      80065      843      85      125720
0.33702825      838      140      77774      842      85      122247
0.3371235      857      140      75112      844      86      118172
0.33721875      883      141      73167      842      85      115469
0.337695      1013      133      86536      842      84      132064
0.33817125      1259      132      135383      852      82      209216
0.3386475      1465      125      148084

In [210]:
amplitude_ry = 0.343029 - 0.3336945
amplitude_ry * 1e3

9.334499999999968

Initialize scan file

In [232]:
filename = 'scan_ry_20250724_01.h5'
filedir  = '/home/ids/data/2025-07-24-Mirror-Ry/'
file = HDF5File(filename=filename,filedir=filedir)

file.save_metadata(metadata_dict={
    'ry': CAX.mirror.ry_pos
})

Execution

In [227]:
ry0 = CAX.mirror.ry_pos

In [ ]:
ry0 = CAX.mirror.ry_pos
print(f'initial ry: {ry0}')

initial ry: 0.33579000000000003


In [ ]:
positions1 = ry0 + np.linspace(0,amplitude_ry,101)[1:]
positions2 = positions1.copy()[-2::-1]
positions = np.hstack([positions1,positions2])

In [ ]:
positions*1e4

array([ 0.93345,  1.8669 ,  2.80035,  3.7338 ,  4.66725,  5.6007 ,
        6.53415,  7.4676 ,  8.40105,  9.3345 , 10.26795, 11.2014 ,
       12.13485, 13.0683 , 14.00175, 14.9352 , 15.86865, 16.8021 ,
       17.73555, 18.669  , 19.60245, 20.5359 , 21.46935, 22.4028 ,
       23.33625, 24.2697 , 25.20315, 26.1366 , 27.07005, 28.0035 ,
       28.93695, 29.8704 , 30.80385, 31.7373 , 32.67075, 33.6042 ,
       34.53765, 35.4711 , 36.40455, 37.338  , 38.27145, 39.2049 ,
       40.13835, 41.0718 , 42.00525, 42.9387 , 43.87215, 44.8056 ,
       45.73905, 46.6725 , 47.60595, 48.5394 , 49.47285, 50.4063 ,
       51.33975, 52.2732 , 53.20665, 54.1401 , 55.07355, 56.007  ,
       56.94045, 57.8739 , 58.80735, 59.7408 , 60.67425, 61.6077 ,
       62.54115, 63.4746 , 64.40805, 65.3415 , 66.27495, 67.2084 ,
       68.14185, 69.0753 , 70.00875, 70.9422 , 71.87565, 72.8091 ,
       73.74255, 74.676  , 75.60945, 76.5429 , 77.47635, 78.4098 ,
       79.34325, 80.2767 , 81.21015, 82.1436 , 83.07705, 84.01

In [ ]:
( positions[1:] - positions[:-1] ) * 1e4

array([ 0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93

In [ ]:
len(positions)*5.8/60

19.236666666666668

In [254]:
0.2366*60

14.196

In [ ]:
t0 = time.time()


for i, pos in enumerate(positions):
    print(f'{i}/{len(positions)-1}')

    move(ry_pos=pos)

    ry = CAX.mirror.ry_pos
    img1 = CAX.dvf_A1.image
    expotime1 = CAX.dvf_A1.exposure_time
    img2 = CAX.dvf_B1.image
    expotime2 = CAX.dvf_B1.exposure_time

    scaname = f'scan-{i:04d}'
    file.save_group(grpname=scaname, grpmetadata={'ry_pos':ry})
    file.save_dataset(grpname=scaname, dsetname='dvf1', 
                      dsetmetadata={'expo_time':expotime1}, dsetdata=img1)
    file.save_dataset(grpname=scaname, dsetname='dvf2', 
                      dsetmetadata={'expo_time':expotime2}, dsetdata=img2)


t1 = time.time()

print()
print(f'elapsed time [min]: {(t1-t0)/60}')

0/198
1/198
2/198
3/198
4/198
5/198
6/198
7/198
8/198
9/198
10/198
11/198
12/198
13/198
14/198
15/198
16/198
17/198
18/198
19/198
20/198
21/198
22/198
23/198
24/198
25/198
26/198
27/198
28/198
29/198
30/198
31/198
32/198
33/198
34/198
35/198
36/198
37/198
38/198
39/198
40/198
41/198
42/198
43/198
44/198
45/198
46/198
47/198
48/198
49/198
50/198
51/198
52/198
53/198
54/198
55/198
56/198
57/198
58/198
59/198
60/198
61/198
62/198
63/198
64/198
65/198
66/198
67/198
68/198
69/198
70/198
71/198
72/198
73/198
74/198
75/198
76/198
77/198
78/198
79/198
80/198
81/198
82/198
83/198
84/198
85/198
86/198
87/198
88/198
89/198
90/198
91/198
92/198
93/198
94/198
95/198
96/198
97/198
98/198
99/198
100/198
101/198
102/198
103/198
104/198
105/198
106/198
107/198
108/198
109/198
110/198
111/198
112/198
113/198
114/198
115/198
116/198
117/198
118/198
119/198
120/198
121/198
122/198
123/198
124/198
125/198
126/198
127/198
128/198
129/198
130/198
131/198
132/198
133/198
134/198
135/198
136/198
137/198
138/19